# Preparazione del dataset



In [1]:
!pip install lxml

## 1. Variazioni testuali di due testimoni manoscritti codificati in TEI-XML



In [2]:
from google.colab import files
uploaded = files.upload()

Saving merged_manuscripts.xml to merged_manuscripts.xml


In [3]:
from lxml import etree
import pandas as pd
import re

tree = etree.parse("merged_manuscripts.xml")
root = tree.getroot()

NS = {"tei": "http://www.tei-c.org/ns/1.0"}

In [4]:
divs = root.findall(".//tei:div", namespaces=NS)

## 2. Estrazione e normalizzazione del testo

Il testo viene estratto dal file XML-TEI preservando il contenuto delle espansioni abbreviative ed escludendo i commenti.
La normalizzazione viene applicata solo alla colonna `text_normalized`, usata per la valutazione computazionale della similarità. La colonna `text_expanded` conserva invece il testo estratto in forma leggibile.

Durante l’estrazione, alcune parole risultano separate perché nel manoscritto sono divise da fine riga e per il confronto computazionale vengono ricomposti selettivamente nella versione normalizzata, senza modificare il testo espanso.

In [5]:
def extract_tei_text(element):
    """
    Extract readable text from a TEI element, preserving text inside
    abbreviation expansions and skipping XML comments.
    """
    parts = []

    if element.text:
        parts.append(element.text)

    for child in element:
        if isinstance(child, etree._Comment):
            if child.tail:
                parts.append(child.tail)
            continue

        parts.append(extract_tei_text(child))

        if child.tail:
            parts.append(child.tail)

    text = "".join(parts)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def normalize_text(text):
    """
    Apply light normalization for computational comparison.

    The original expanded text is not modified. This normalized version is used
    only for similarity evaluation.
    """
    text = text.lower()

    # Recompose selected word splits caused by line breaks or word division
    # in the manuscript/transcription. These replacements affect only the
    # normalized text used for computation, not the expanded text.
    line_break_recompositions = {
        "con cilium": "concilium",
        "difficul tates": "difficultates",
        "obe diunt": "obediunt",
        "im perator": "imperator"
    }

    for split_form, recomposed_form in line_break_recompositions.items():
        text = text.replace(split_form, recomposed_form)

    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

Verifico che effettivamente la funzione `normalize_text` stia ricomponendo correttamente le parole divise.




In [6]:
normalization_examples = {
    "con cilium": normalize_text("con cilium"),
    "difficul tates": normalize_text("difficul tates"),
    "obe diunt": normalize_text("obe diunt"),
    "Im perator": normalize_text("Im perator")
}

normalization_examples

{'con cilium': 'concilium',
 'difficul tates': 'difficultates',
 'obe diunt': 'obediunt',
 'Im perator': 'imperator'}

## 3. Creazione del dataset

In [7]:
section_titles = {
    "wit_D54": "Nota quod Sacro concilio non est detrahendum patet ex multis",
    "wit_4941": "De sacro Concilio non est detrahendum patet ex multis"
}

manuscript_labels = {
    "wit_D54": "Prague D.54",
    "wit_4941": "Wien 4941"
}

rows = []

for d in divs:
    witness = d.get("{http://www.w3.org/XML/1998/namespace}id")
    manuscript = manuscript_labels.get(witness, witness)
    section_title = section_titles.get(witness, "")

    first_p = d.find(".//tei:p", namespaces=NS)
    segments = first_p.findall(".//tei:seg[@type='argument']", namespaces=NS)

    for seg in segments:
        text_expanded = extract_tei_text(seg)
        text_normalized = normalize_text(text_expanded)

        rows.append({
            "witness": witness,
            "manuscript": manuscript,
            "section_title": section_title,
            "argument_n": seg.get("n"),
            "text_expanded": text_expanded,
            "text_normalized": text_normalized
        })

df_segments = pd.DataFrame(rows)
df_segments

,witness,manuscript,section_title,argument_n,text_expanded,text_normalized
0,wit_D54,Prague D.54,Nota quod Sacro concilio non est detrahendum p...,1,primo apostoli illud in instituunt et celebravunt,primo apostoli illud in instituunt et celebravunt
1,wit_D54,Prague D.54,Nota quod Sacro concilio non est detrahendum p...,2,Secundo quia per sacrorum concilium quattuor s...,secundo quia per sacrorum concilium quattuor s...
2,wit_D54,Prague D.54,Nota quod Sacro concilio non est detrahendum p...,3,Tercio Sacrum con cilium solet terminare omnes...,tercio sacrum concilium solet terminare omnes ...
3,wit_D54,Prague D.54,Nota quod Sacro concilio non est detrahendum p...,4,Quarto quia papa et Imperator confirmarunt con...,quarto quia papa et imperator confirmarunt con...
4,wit_D54,Prague D.54,Nota quod Sacro concilio non est detrahendum p...,5,Quinto quia omnes Reges christianitatis et omn...,quinto quia omnes reges christianitatis et omn...
5,wit_D54,Prague D.54,Nota quod Sacro concilio non est detrahendum p...,6,Sexto quia nullus tam magnus est qui presumat ...,sexto quia nullus tam magnus est qui presumat ...
6,wit_4941,Wien 4941,De sacro Concilio non est detrahendum patet ex...,1,Primo quia Apostoli ide instituerunt et celebr...,primo quia apostoli ide instituerunt et celebr...
7,wit_4941,Wien 4941,De sacro Concilio non est detrahendum patet ex...,2,secundo quia per sacrum Concilium quator sanct...,secundo quia per sacrum concilium quator sanct...
8,wit_4941,Wien 4941,De sacro Concilio non est detrahendum patet ex...,3,Tercio sacrum Concilium solet terminare omnes ...,tercio sacrum concilium solet terminare omnes ...
9,wit_4941,Wien 4941,De sacro Concilio non est detrahendum patet ex...,4,Quarto quia papa et Imperator confirmaventur C...,quarto quia papa et imperator confirmaventur c...


## 4. Preview ed esportazione

In [8]:
df_segments[["manuscript", "argument_n", "text_expanded", "text_normalized"]]

,manuscript,argument_n,text_expanded,text_normalized
0,Prague D.54,1,primo apostoli illud in instituunt et celebravunt,primo apostoli illud in instituunt et celebravunt
1,Prague D.54,2,Secundo quia per sacrorum concilium quattuor s...,secundo quia per sacrorum concilium quattuor s...
2,Prague D.54,3,Tercio Sacrum con cilium solet terminare omnes...,tercio sacrum concilium solet terminare omnes ...
3,Prague D.54,4,Quarto quia papa et Imperator confirmarunt con...,quarto quia papa et imperator confirmarunt con...
4,Prague D.54,5,Quinto quia omnes Reges christianitatis et omn...,quinto quia omnes reges christianitatis et omn...
5,Prague D.54,6,Sexto quia nullus tam magnus est qui presumat ...,sexto quia nullus tam magnus est qui presumat ...
6,Wien 4941,1,Primo quia Apostoli ide instituerunt et celebr...,primo quia apostoli ide instituerunt et celebr...
7,Wien 4941,2,secundo quia per sacrum Concilium quator sanct...,secundo quia per sacrum concilium quator sanct...
8,Wien 4941,3,Tercio sacrum Concilium solet terminare omnes ...,tercio sacrum concilium solet terminare omnes ...
9,Wien 4941,4,Quarto quia papa et Imperator confirmaventur C...,quarto quia papa et imperator confirmaventur c...


In [9]:
df_segments.to_csv("segments_dataset.csv", index=False, encoding="utf-8")
files.download("segments_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Similarity Evaluation

Questa sezione confronta il testo delle sei argomentazioni nei due testimoni (Prague D.54 e Wien 4941), quindi a livello degli elementi TEI `<seg type="argument">`.

Vengono utilizzate due misure di similarità: **Jaccard similarity**, che valuta la sovrapposizione lessicale indipendentemente dall’ordine delle parole, e la **Longest Common Subsequence similarity**, che valuta quanto del materiale testuale compare nello stesso ordine nei due testimoni.

Le misure di similarità sono calcolate sulla colonna `text_normalized`, non sulla colonna `text_expanded`. Questo permette di ridurre l’effetto computazionale di fenomeni grafici come la divisione di parola a fine riga, mantenendo comunque separato il testo estratto dalla sua forma normalizzata.

In [10]:
def jaccard_similarity(text_a, text_b):
    """
    Compute Jaccard similarity between two texts.
    The texts are represented as sets of tokens.
    """
    A = set(text_a.split())
    B = set(text_b.split())

    intersection = A & B
    union = A | B

    if len(union) == 0:
        return 1.0

    return len(intersection) / len(union)


def longest_common_subsequence(seq_a, seq_b):
    """
    Compute the Longest Common Subsequence between two token sequences.
    """
    m = len(seq_a)
    n = len(seq_b)

    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if seq_a[i - 1] == seq_b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])

    # Reconstruct LCS
    i, j = m, n
    lcs = []

    while i > 0 and j > 0:
        if seq_a[i - 1] == seq_b[j - 1]:
            lcs.append(seq_a[i - 1])
            i -= 1
            j -= 1
        elif dp[i - 1][j] >= dp[i][j - 1]:
            i -= 1
        else:
            j -= 1

    lcs.reverse()
    return lcs


def lcs_similarity(text_a, text_b):
    """
    Compute normalized LCS similarity between two texts.
    The score is the length of the LCS divided by the maximum sequence length.
    """
    seq_a = text_a.split()
    seq_b = text_b.split()

    if len(seq_a) == 0 and len(seq_b) == 0:
        return 1.0

    lcs = longest_common_subsequence(seq_a, seq_b)
    return len(lcs) / max(len(seq_a), len(seq_b))

In [11]:
similarity_rows = []

for argument_n in sorted(df_segments["argument_n"].unique(), key=int):

    d54_text = df_segments[
        (df_segments["witness"] == "wit_D54") &
        (df_segments["argument_n"] == argument_n)
    ]["text_normalized"].iloc[0]

    wien_text = df_segments[
        (df_segments["witness"] == "wit_4941") &
        (df_segments["argument_n"] == argument_n)
    ]["text_normalized"].iloc[0]

    jaccard = jaccard_similarity(d54_text, wien_text)
    lcs = lcs_similarity(d54_text, wien_text)

    similarity_rows.append({
        "argument_n": argument_n,
        "jaccard_similarity": round(jaccard, 3),
        "lcs_similarity": round(lcs, 3),
        "D54_length_tokens": len(d54_text.split()),
        "4941_length_tokens": len(wien_text.split())
    })

df_similarity = pd.DataFrame(similarity_rows)
df_similarity

,argument_n,jaccard_similarity,lcs_similarity,D54_length_tokens,4941_length_tokens
0,1,0.273,0.429,7,7
1,2,0.760,0.857,28,28
2,3,0.875,0.609,23,14
3,4,0.882,0.944,18,18
4,5,0.632,0.783,20,23
5,6,0.913,0.909,22,22


In [12]:
def interpret_similarity(jaccard, lcs):
    """
    Provide a simple qualitative interpretation of similarity scores.
    """
    if jaccard >= 0.75 and lcs >= 0.75:
        return "high lexical and sequential similarity"
    elif jaccard >= 0.75 and lcs < 0.75:
        return "high lexical overlap, lower sequential similarity"
    elif jaccard < 0.75 and lcs >= 0.75:
        return "lower lexical overlap, but stable sequence"
    else:
        return "lower similarity; possible omission, addition, or reformulation"


df_similarity["interpretation"] = df_similarity.apply(
    lambda row: interpret_similarity(
        row["jaccard_similarity"],
        row["lcs_similarity"]
    ),
    axis=1
)

df_similarity

,argument_n,jaccard_similarity,lcs_similarity,D54_length_tokens,4941_length_tokens,interpretation
0,1,0.273,0.429,7,7,"lower similarity; possible omission, addition,..."
1,2,0.760,0.857,28,28,high lexical and sequential similarity
2,3,0.875,0.609,23,14,"high lexical overlap, lower sequential similarity"
3,4,0.882,0.944,18,18,high lexical and sequential similarity
4,5,0.632,0.783,20,23,"lower lexical overlap, but stable sequence"
5,6,0.913,0.909,22,22,high lexical and sequential similarity


In [13]:
df_similarity.to_csv("similarity_by_segment.csv", index=False, encoding="utf-8")
files.download("similarity_by_segment.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 6. Interpretazione dei casi selezionati

I punteggi che seguono servono a selezionare alcuni casi significativi, più o meno stabili, da osservare manualmente per indagare la variazione tra i testimoni.


In [14]:
comparison_rows = []

for argument_n in sorted(df_segments["argument_n"].unique(), key=int):

    d54_row = df_segments[
        (df_segments["witness"] == "wit_D54") &
        (df_segments["argument_n"] == argument_n)
    ].iloc[0]

    wien_row = df_segments[
        (df_segments["witness"] == "wit_4941") &
        (df_segments["argument_n"] == argument_n)
    ].iloc[0]

    similarity_row = df_similarity[
        df_similarity["argument_n"] == argument_n
    ].iloc[0]

    comparison_rows.append({
        "argument_n": argument_n,
        "D54_text": d54_row["text_normalized"],
        "4941_text": wien_row["text_normalized"],
        "jaccard_similarity": similarity_row["jaccard_similarity"],
        "lcs_similarity": similarity_row["lcs_similarity"],
        "interpretation": similarity_row["interpretation"]
    })

df_comparison = pd.DataFrame(comparison_rows)
df_comparison

,argument_n,D54_text,4941_text,jaccard_similarity,lcs_similarity,interpretation
0,1,primo apostoli illud in instituunt et celebravunt,primo quia apostoli ide instituerunt et celebr...,0.273,0.429,"lower similarity; possible omission, addition,..."
1,2,secundo quia per sacrorum concilium quattuor s...,secundo quia per sacrum concilium quator sanct...,0.760,0.857,high lexical and sequential similarity
2,3,tercio sacrum concilium solet terminare omnes ...,tercio sacrum concilium solet terminare omnes ...,0.875,0.609,"high lexical overlap, lower sequential similarity"
3,4,quarto quia papa et imperator confirmarunt con...,quarto quia papa et imperator confirmaventur c...,0.882,0.944,high lexical and sequential similarity
4,5,quinto quia omnes reges christianitatis et omn...,quinto quia omnes reges christianitatis et omn...,0.632,0.783,"lower lexical overlap, but stable sequence"
5,6,sexto quia nullus tam magnus est qui presumat ...,sexto quia nullus est tam magnus qui presumat ...,0.913,0.909,high lexical and sequential similarity


## 6.1 Analisi dei segmenti più stabili (6, 4 e 2)

Analizzando la tabella vediamo che i segmenti più stabili, e quindi con valori più alti, sono 6, 4 e 2.

L’argomento 6 presenta il valore più alto della Jaccard similarity (0.913) e un valore molto alto di LCS similarity (0.909). Questo indica che i due testimoni condividono quasi lo stesso materiale lessicale (notiamo la variante grafica *concilium/consilium*) e mantengono una sequenza testuale molto simile (*est* prima e dopo *tam magnus*), con pochissime differenze che non alterano la struttura complessiva del segmento.

L’argomento 4 presenta invece il valore più alto di LCS similarity (0.944), insieme a una Jaccard similarity molto alta (0.882); questo perchè i due testimoni conservano lo stesso ordine testuale, cambia solo il tempo verbale *confirmarunt/confirmaventur*.

Anche l’argomento 2 mostra valori abbastanza alti in entrambe le misure: Jaccard = 0.760 e LCS = 0.857. In questo caso, la sequenza rimane intatta, cambiano le varianti grafiche o morfologiche.

## 6.2 Analisi dei segmenti più variabili (1, 3 e 5)

Gli argomenti 1, 3 e 5 mostrano risultati più problematici o meno uniformi, ma per ragioni diverse.

Quello con i valori più bassi è l'argomento 1: Jaccard similarity = 0.273 e LCS similarity = 0.429. Tuttavia, questo segmento è molto breve, con 7 token in entrambi i testimoni. Di conseguenza, le misure sono particolarmente sensibili anche a piccole variazioni. La differenza tra forme come *instituunt/instituerunt* e *celebravunt/celebrarunt* pesano sul punteggio abbassandolo.

L’argomento 3 è particolarmente interessante perché presenta una Jaccard similarity alta (0.875), ma una LCS similarity più bassa (0.609). Questo significa che i due testimoni condividono molto materiale lessicale, ma non lo conservano nella stessa estensione sequenziale. Inoltre, la differenza di lunghezza è significativa: D54 contiene 23 token, mentre 4941 ne contiene 14 poichè include anche la parte *et dampnare omnes hereses que surgunt in ecclesia dei*. Il risultato suggerisce quindi una possibile omissione o compressione in 4941.

L’argomento 5 mostra un caso diverso: la Jaccard similarity è più bassa (0.632), mentre la LCS similarity è relativamente alta (0.783). Questo suggerisce che la sequenza resta riconoscibile, ma con differenze lessicali e con materiale aggiuntivo in 4941: il troncamento di *episcopi* nel testimone 1 e l'aggiunta della formula finale *quod nos obediamus* nel testimone 2.

In [15]:
df_comparison.to_csv("comparison_by_segment.csv", index=False, encoding="utf-8")
files.download("comparison_by_segment.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>